In [ ]:
# Setup (added for the release): the notebook's folder on sys.path, then work in its data folder.
# The curation ran with the current directory = that folder; see curation/README.md for its contents.
import os, sys
if "data/curation/" not in os.getcwd().replace(os.sep, "/") + "/":
    sys.path.insert(0, os.path.abspath("."))
    os.chdir('../../data/curation/whole_BMRB/cluster_30p')
print("working folder:", os.getcwd())


# Tag and linker labels

`assign_labels.relabel_tags` labels each entry from its sequence alone and the labels are written to `tag_labels.csv`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from assign_labels import relabel_tags

In [ ]:
PANAV_DIR = Path(
    "../../rBMRB/cluster_30p/PANAV"
)
NOREF_Whole_DIR = Path(
    "../../whole_BMRB/cluster_30p"
)
JSON_FILES = ("train.json", "val.json", "test.json")
OUT_CSV = Path(
    "../../whole_BMRB/cluster_30p/tag_labels.csv"
)

## Check a few entries

`T` or `t` in the middle of a sequence would indicate a false motif hit.

In [ ]:
with open(PANAV_DIR / "val.json") as f:
    _val = json.load(f)

for ref_id in list(_val["sequence"].keys())[:10]:
    seq = _val["sequence"][ref_id]
    entry_id = _val["entry_ID"][ref_id]
    label = relabel_tags(seq)
    n_tag, n_flank = label.count("T"), label.count("t")
    print(f"{entry_id:14s} len={len(seq):4d}  tag={n_tag:3d}  flank={n_flank:3d}")
    if n_tag or n_flank:
        print("  seq  ", seq)
        print("  label", label)

## All entries

An entry found in several json files is kept once.

In [ ]:
rows = {}  # entry_ID -> row
for name in JSON_FILES:
    if name == "train.json":
        with open(NOREF_Whole_DIR / name) as f:
            d = json.load(f)    
    else:    
        with open(PANAV_DIR / name) as f:
            d = json.load(f)

    for ref_id, entry_id in d["entry_ID"].items():
        if entry_id in rows:
            continue
        seq = d["sequence"][ref_id]
        label = relabel_tags(seq)

        rows[entry_id] = {
            # lower-cased, "bmr" stripped, suffix kept -- matches
            # chemical_shift_trainer_noref.py's pdb_id convention
            # (_json_to_dict: entry_ID.lower().removeprefix("bmr")).
            "BMRB_entry": entry_id.lower().removeprefix("bmr"),
            "sequence": seq,
            "label": label,
            "seq_len": len(seq),
            "tag": label.count("T"),
            "flank": label.count("t"),
        }

df = pd.DataFrame(rows.values())
print(f"{len(df)} unique entries")
df.head()

## Tagged fraction

A large fraction can mean that a short cleavage motif matched the native sequence near a terminus.

In [ ]:
frac_tagged = (df["tag"] + df["flank"]) / df["seq_len"]
print(frac_tagged.describe())
df.assign(frac_tagged=frac_tagged).sort_values("frac_tagged", ascending=False).head(20)

In [ ]:
df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(df)} rows to {OUT_CSV.resolve()}")